# O'Bara — Banc d'essai ASR

**Plateforme : Kaggle.** Paramètres du notebook → *Accelerator* : **GPU T4 x1** ·
*Internet* : **On** · *Secrets* : ajouter `HF_TOKEN` (lecture, conditions acceptées sur
`uvci/koumankan4dyula` **et** `abdouaziiz/baoule` — les deux sont à accès restreint). Puis **Save Version → Save & Run All** : le calcul
continue navigateur fermé, et le fichier de résultats devient une sortie versionnée.

**Deux passes, deux environnements.** Omnilingual ASR remplace la version de PyTorch
préinstallée ; les modèles Hugging Face (Whisper, wav2vec2) utilisent celle de Kaggle.
On ne les mélange pas dans une même session : lancer ce notebook **deux fois**, une avec
`FAMILLE = "transformers"`, une avec `FAMILLE = "omnilingual"`. Les deux fichiers de
résultats se fusionnent ensuite avec `python bench/rapport_asr.py` dans le dépôt.

Protocole complet : compétence `obara-banc-essai` et `docs/BANC-ESSAI.md`.

In [ ]:
# ═══════════ PARAMÈTRES ═══════════
FAMILLE = "transformers"      # "transformers" puis "omnilingual" — une famille par exécution
N_PAR_CORPUS = 60             # 60 énoncés = CER médian stable à ±3 points
DUREE_MAX_S = 38              # Omnilingual refuse l'audio > 40 s
CORPUS_ACTIFS = ["BCI-CV", "BCI-Z", "DYU-K", "BAM-J", "TERRAIN"]
CORPUS_PUBLICS = {"BCI-CV", "BCI-Z", "BAM-J"}   # seuls corpus dont exemples et locuteurs vont dans le JSON publiable ;
                                                # TERRAIN (voix réelles) et DYU-K (accès restreint) restent privés

In [ ]:
import os, sys, subprocess
def installer(*p): subprocess.run([sys.executable, "-m", "pip", "install", "-q", *p], check=True)
installer("jiwer", "librosa", "soundfile", "datasets>=2.19")
if FAMILLE == "omnilingual":
    # omnilingual-asr épingle torch 2.8.0 mais laisse torchaudio libre : pip installe alors un
    # torchaudio récent compilé pour CUDA 13 → « libcudart.so.13 introuvable ». On l'épingle.
    installer("omnilingual-asr", "torchaudio==2.8.0")   # remplace torch : jamais transformers dans la même session
print("installation terminée pour", FAMILLE)

In [ ]:
import os
# Jeton Hugging Face : secrets Kaggle, sinon Colab, sinon variable d'environnement
HF_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    try:
        from google.colab import userdata; HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        HF_TOKEN = os.environ.get("HF_TOKEN")
print("jeton HF :", "présent" if HF_TOKEN else "ABSENT — le corpus dioula DYU-K sera ignoré")
SORTIE = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."

In [ ]:
# Matériel réel de cette session — et quels modèles Omnilingual peuvent y tenir.
# Tailles de checkpoint MESURÉES sur les cartes officielles (fp32). Il faut au moins cette
# quantité de RAM pour charger, puis environ la moitié en VRAM en float16.
import shutil, torch
ram = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1e9
vram = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "aucun"
print(f"GPU : {gpu} · VRAM {vram:.1f} Go · RAM {ram:.1f} Go · disque libre {shutil.disk_usage('/').free/1e9:.0f} Go")
CHECKPOINT_GO = {"omniASR_CTC_300M_v2": 1.30, "omniASR_CTC_1B_v2": 3.90, "omniASR_LLM_300M_v2": 6.53,
                 "omniASR_LLM_1B_v2": 9.12, "omniASR_LLM_3B_v2": 17.52, "omniASR_LLM_7B_v2": 31.22}
TROP_GROS = set()
for m, go in CHECKPOINT_GO.items():
    ok_ram, ok_vram = go * 1.15 < ram, (go / 2) * 1.2 < vram or vram == 0
    if not (ok_ram and ok_vram): TROP_GROS.add(m)
    print(f"  {m:22s} {go:5.1f} Go  →  {'OK' if ok_ram and ok_vram else 'NE TIENT PAS (' + ('RAM' if not ok_ram else 'VRAM') + ')'}")

## Normalisation — identique pour la référence et l'hypothèse

Les tons et diacritiques sont **conservés** (ils sont distinctifs en baoulé et en
dioula). Les variantes d'apostrophe (`'` `’` `ʼ`) sont unifiées : les corpus baoulé en
mélangent trois. Les numéros de versets en tête de ligne (39 % du corpus Zenodo) sont
retirés — un moteur ASR ne les prononce pas.

In [ ]:
import re, unicodedata
APOS = dict.fromkeys(map(ord, "’ʼ‘`´"), "'")
def normaliser(t):
    t = unicodedata.normalize("NFC", str(t)).translate(APOS).lower()
    t = re.sub(r"^\s*[\d?]+\s+", "", t)                   # numéro de verset en tête
    t = re.sub(r"[“”«»\"]", " ", t)
    t = re.sub(r"[^\w\s']", " ", t)                        # ponctuation, sauf apostrophe
    t = re.sub(r"(?<!\w)'|'(?!\w)", " ", t)                # apostrophe isolée
    return re.sub(r"\s+", " ", t).strip()
assert normaliser("37  Zezi seli i kɛ: “A wunnin i.”") == "zezi seli i kɛ a wunnin i"
assert normaliser("Bakannganʼm be su") == "bakanngan'm be su"
print("normalisation ok")

## Corpus

| Code | Langue | Pourquoi il est là | Limite connue |
|---|---|---|---|
| `BCI-CV` | baoulé | **Référence baoulé** : split test à 9 locuteurs, aucun partagé avec l'entraînement | phrases lues |
| `BCI-Z` | baoulé | Contrôle d'une autre origine (data354, Abidjan 2022) | **4 locuteurs présents dans train ET test**, 70 % une seule voix, 39 % de versets bibliques |
| `DYU-K` | dioula | Référence dioula (UVCI + data354) | accès sur conditions |
| `BAM-J` | bambara | Contrôle manding | pas la langue cible |
| `TERRAIN` | dioula / baoulé / fr | **Le seul corpus réaliste** : vraies annonces, vrai bruit | à constituer — voir `bench/corpus_terrain/` |

L'échantillonnage tourne entre les locuteurs pour qu'une voix dominante n'écrase pas la
mesure.

In [ ]:
import io, zipfile, csv, urllib.request, itertools, collections, subprocess
import numpy as np, soundfile as sf, librosa
from datasets import load_dataset, Audio

def decoder(octets=None, chemin=None, sr=16000):
    """WAV/FLAC/MP3 via soundfile ; m4a/aac (corpus Zenodo) via ffmpeg — sans passer par
    audioread, déprécié dans librosa et qui casserait en silence à la version 1.0."""
    try:
        x, s = sf.read(io.BytesIO(octets) if octets is not None else chemin, dtype="float32")
    except Exception:
        cmd = ["ffmpeg", "-v", "quiet", "-i", "pipe:0" if octets is not None else chemin,
               "-f", "wav", "-ac", "1", "-ar", str(sr), "pipe:1"]
        out = subprocess.run(cmd, input=octets, capture_output=True, check=True).stdout
        x, s = sf.read(io.BytesIO(out), dtype="float32")
    if x.ndim > 1: x = x.mean(axis=1)
    return (librosa.resample(x, orig_sr=s, target_sr=sr) if s != sr else x).astype("float32")

def equilibrer(items, n):
    """Tourne entre locuteurs : chaque voix contribue à parts égales tant qu'elle a des énoncés."""
    par = collections.defaultdict(list)
    for it in items: par[it["locuteur"]].append(it)
    out = []
    for lot in itertools.zip_longest(*par.values()):
        out += [x for x in lot if x]
        if len(out) >= n: break
    return out[:n]

def garder(x, ref, loc, id_):
    d = len(x) / 16000
    return {"audio": x, "ref": ref, "locuteur": loc, "duree": d, "id": id_} if 0.8 < d <= DUREE_MAX_S and normaliser(ref) else None

def charger_bci_cv():
    ds = load_dataset("Klayt/baoule-common-voice", split="test").cast_column("audio", Audio(decode=False))
    items = [garder(decoder(r["audio"]["bytes"]), r["sentence"], r["client_id"][:10], r["path"]) for r in ds]
    return "bci", equilibrer([i for i in items if i], N_PAR_CORPUS)

def charger_bci_z():
    dest = "/tmp/bci.zip"
    if not os.path.exists(dest):
        urllib.request.urlretrieve("https://zenodo.org/records/6705861/files/bci-datasets.zip?download=1", dest)
    z = zipfile.ZipFile(dest); z.extractall("/tmp")
    lignes = list(csv.reader(open("/tmp/bci-datasets/test.tsv", encoding="utf-8"), delimiter="\t"))[1:]
    items = []
    for fid, texte in (l for l in lignes if len(l) >= 2):
        p = f"/tmp/bci-datasets/wav/{fid}"
        if os.path.exists(p):
            items.append(garder(decoder(chemin=p), texte, re.match(r"(OD_[MF]\d+)", fid).group(1), fid))
    return "bci", equilibrer([i for i in items if i], N_PAR_CORPUS)

def _colonne(ds, indices):
    return next(c for c in ds.column_names if c != "audio" and any(k in c.lower() for k in indices))

def charger_dyu_k():
    if not HF_TOKEN: return "dyu", []
    ds = load_dataset("uvci/koumankan4dyula", split="test", token=HF_TOKEN).cast_column("audio", Audio(decode=False))
    col = _colonne(ds, ["dyu", "dyula", "transcri", "sentence", "text"])
    loc_col = next((c for c in ds.column_names if "speaker" in c.lower() or "locuteur" in c.lower()), None)
    items = [garder(decoder(r["audio"]["bytes"]), r[col], str(r[loc_col]) if loc_col else "?", str(i)) for i, r in enumerate(ds)]
    return "dyu", equilibrer([i for i in items if i], N_PAR_CORPUS)

def charger_bam_j():
    ds = load_dataset("RobotsMali/jeli-asr", split="test").cast_column("audio", Audio(decode=False))
    col = _colonne(ds, ["bam", "bambara", "transcri", "text", "sentence"])
    items = [garder(decoder(r["audio"]["bytes"]), r[col], "?", str(i)) for i, r in enumerate(itertools.islice(ds, 400))]
    return "bam", [i for i in items if i][:N_PAR_CORPUS]

def charger_terrain():
    base = next((p for p in ["/kaggle/input/obara-corpus-terrain", "bench/corpus_terrain", "corpus_terrain"] if os.path.exists(f"{p}/manifeste.tsv")), None)
    if not base: return "mixte", []
    lignes = list(csv.DictReader(open(f"{base}/manifeste.tsv", encoding="utf-8"), delimiter="\t"))
    items = [dict(garder(decoder(chemin=f"{base}/audio/{l['fichier']}"), l["texte"], l["locuteur"], l["fichier"]) or {}, langue=l["langue"])
             for l in lignes]
    return "mixte", [i for i in items if "audio" in i]

CHARGEURS = {"BCI-CV": charger_bci_cv, "BCI-Z": charger_bci_z, "DYU-K": charger_dyu_k, "BAM-J": charger_bam_j, "TERRAIN": charger_terrain}
CORPUS = {}
for nom in CORPUS_ACTIFS:
    try:
        langue, items = CHARGEURS[nom]()
        CORPUS[nom] = {"langue": langue, "items": items}
        print(f"{nom:8s} {langue:5s} {len(items):3d} énoncés · {len({i['locuteur'] for i in items})} locuteurs")
    except Exception as ex:
        # Pour un corpus privé, le message pourrait contenir un nom de fichier ou de locuteur.
        print(f"{nom:8s} IGNORÉ — {type(ex).__name__}" + (f": {str(ex)[:120]}" if nom in CORPUS_PUBLICS else ""))

## Modèles

Chaque entrée porte sa **licence vérifiée sur la fiche**. Un modèle sans licence déclarée
est mesuré mais ne peut pas être retenu pour la production.

In [ ]:
LANG = {"dyu": "dyu_Latn", "bci": "bci_Latn", "bam": "bam_Latn"}
MODELES = {
    # famille transformers — PyTorch de Kaggle
    "abdouaziiz/baoule":                 {"famille": "transformers", "langues": ["bci"], "licence": "Apache-2.0",
                                          "note": "corpus d'entraînement non public : suspect de fuite sur BCI-Z"},
    "Klayt/wav2vec2-large-xlsr-baoule-demo": {"famille": "transformers", "langues": ["bci"], "licence": "Apache-2.0"},
    "Lingua-Africa/whisper-baoule-small": {"famille": "transformers", "langues": ["bci"], "licence": "non déclarée"},
    # famille omnilingual — environnement isolé
    "omniASR_LLM_300M_v2": {"famille": "omnilingual", "langues": ["dyu", "bci", "bam"], "licence": "Apache-2.0"},
    "omniASR_LLM_1B_v2":   {"famille": "omnilingual", "langues": ["dyu", "bci", "bam"], "licence": "Apache-2.0"},
    "omniASR_LLM_3B_v2":   {"famille": "omnilingual", "langues": ["dyu", "bci", "bam"], "licence": "Apache-2.0",
                            "note": "~10 Go de VRAM : tient sur une T4"},
    "omniASR_CTC_300M_v2": {"famille": "omnilingual", "langues": ["dyu", "bci", "bam"], "licence": "Apache-2.0",
                            "note": "CTC : ignore l'étiquette de langue, sert de repère de vitesse"},
}
A_EVALUER = [m for m, v in MODELES.items() if v["famille"] == FAMILLE and m not in TROP_GROS]
if TROP_GROS & set(MODELES): print("ignorés faute de mémoire :", sorted(TROP_GROS & set(MODELES)))
print(f"{len(A_EVALUER)} modèles pour cette exécution :", A_EVALUER)

In [ ]:
import gc, time, torch

def charger_moteur(nom):
    spec = MODELES[nom]
    if spec["famille"] == "transformers":
        from transformers import pipeline
        p = pipeline("automatic-speech-recognition", model=nom, device=0 if torch.cuda.is_available() else -1,
                     token=HF_TOKEN)
        return lambda audio, langue: p({"raw": audio, "sampling_rate": 16000})["text"], p
    from omnilingual_asr.models.inference.pipeline import ASRInferencePipeline
    # Le pipeline charge en bfloat16 par défaut. La T4 de Kaggle (Turing) ne le supporte pas
    # nativement : on passe en float16 sur GPU ancien, float32 sur CPU.
    if torch.cuda.is_available():
        dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    else:
        dtype = torch.float32
    p = ASRInferencePipeline(model_card=nom, dtype=dtype)
    MODELES[nom]["dtype"] = str(dtype).replace("torch.", "")
    def f(audio, langue):
        chemin = "/tmp/_o.wav"; sf.write(chemin, audio, 16000)
        return p.transcribe([chemin], lang=[LANG.get(langue)], batch_size=1)[0]
    return f, p

import jiwer, statistics, datetime, json
RESULTATS = []
EXEMPLES_PRIVES = {}   # exemples des corpus privés, écrits à part (cellule du verdict)
for nom in A_EVALUER:
    print(f"\n══ {nom}")
    try:
        torch.cuda.reset_peak_memory_stats() if torch.cuda.is_available() else None
        transcrire, objet = charger_moteur(nom)
    except Exception as ex:
        print("   CHARGEMENT IMPOSSIBLE :", type(ex).__name__, str(ex)[:160]); continue
    for cnom, c in CORPUS.items():
        cibles = [i for i in c["items"] if (i.get("langue") or c["langue"]) in MODELES[nom]["langues"]]
        if not cibles: continue
        lignes = []
        for it in cibles:
            t0 = time.time()
            try: hyp = transcrire(it["audio"], it.get("langue") or c["langue"])
            except Exception as ex: hyp = ""
            ms = (time.time() - t0) * 1000
            r, h = normaliser(it["ref"]), normaliser(hyp)
            lignes.append({"id": it["id"], "locuteur": it["locuteur"], "ref": r, "hyp": h,
                           "cer": jiwer.cer(r, h) if h else 1.0, "wer": jiwer.wer(r, h) if h else 1.0,
                           "latence_ms": round(ms), "rtf": ms / 1000 / it["duree"], "echec": not h})
        par_loc = {l: statistics.median(x["cer"] for x in lignes if x["locuteur"] == l) for l in {x["locuteur"] for x in lignes}}
        res = {"modele": nom, "licence": MODELES[nom]["licence"], "corpus": cnom, "langue": c["langue"], "n": len(lignes),
               "cer_median": statistics.median(x["cer"] for x in lignes), "wer_median": statistics.median(x["wer"] for x in lignes),
               "latence_ms_mediane": round(statistics.median(x["latence_ms"] for x in lignes)),
               "rtf_median": round(statistics.median(x["rtf"] for x in lignes), 3),
               "taux_echec": sum(x["echec"] for x in lignes) / len(lignes),
               "cer_par_locuteur": par_loc if cnom in CORPUS_PUBLICS else {},
               "exemples": lignes[:8] if cnom in CORPUS_PUBLICS else []}
        RESULTATS.append(res)
        if cnom not in CORPUS_PUBLICS:
            EXEMPLES_PRIVES[f"{nom} | {cnom}"] = {"cer_par_locuteur": par_loc, "exemples": lignes[:8]}
        print(f"   {cnom:8s} CER {res['cer_median']:6.1%}  WER {res['wer_median']:6.1%}  "
              f"{res['latence_ms_mediane']:5d} ms  RTF {res['rtf_median']:.2f}  échecs {res['taux_echec']:.0%}")
    vram = torch.cuda.max_memory_allocated() / 2**30 if torch.cuda.is_available() else 0
    for r in RESULTATS:
        if r["modele"] == nom: r["vram_pic_go"] = round(vram, 1)
    print(f"   VRAM pic : {vram:.1f} Go")
    del objet, transcrire; gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

## Lecture croisée

Pour un modèle mesuré sur deux corpus de la même langue, un écart de CER supérieur à
**30 points** signale une fuite présumée : le modèle a vu l'un des deux à l'entraînement.
On retient alors le **moins bon** score comme estimation réaliste.

In [ ]:
SEUIL_CER, SEUIL_FUITE = 0.25, 0.30
par_modele = collections.defaultdict(dict)
for r in RESULTATS: par_modele[r["modele"]][r["corpus"]] = r
print(f"{'modèle':40s} {'BCI-CV':>8s} {'BCI-Z':>8s} {'écart':>7s} {'DYU-K':>8s}  verdict")
for m, cs in par_modele.items():
    g = lambda k: cs[k]["cer_median"] if k in cs else None
    a, b, d = g("BCI-CV"), g("BCI-Z"), g("DYU-K")
    ecart = abs(a - b) if a is not None and b is not None else None
    pires = [x for x in (a, b, d) if x is not None]
    realiste = max(pires) if ecart and ecart > SEUIL_FUITE else (min(pires) if pires else None)
    if ecart is not None and ecart > SEUIL_FUITE: v = "FUITE PRÉSUMÉE"
    elif realiste is not None and realiste < SEUIL_CER and MODELES[m]["licence"] != "non déclarée": v = "RETENU"
    elif realiste is not None and realiste < SEUIL_CER: v = "en réserve (licence)"
    else: v = "écarté"
    f = lambda x: f"{x:7.1%}" if x is not None else "      —"
    print(f"{m:40s} {f(a)} {f(b)} {f(ecart)} {f(d)}  {v}")
    for r in cs.values(): r["verdict"] = v

horodatage = datetime.datetime.now(datetime.timezone.utc).strftime("%Y-%m-%dT%H%M")
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
chemin = f"{SORTIE}/{horodatage[:10]}_asr_{FAMILLE}.json"
json.dump({"etage": "asr", "famille": FAMILLE, "date": horodatage, "gpu": gpu, "n_par_corpus": N_PAR_CORPUS,
           "corpus": {k: {"langue": v["langue"], "n": len(v["items"])} for k, v in CORPUS.items()},
           "resultats": RESULTATS}, open(chemin, "w", encoding="utf-8"), ensure_ascii=False, indent=1)
print("\nrésultats →", chemin, "| GPU :", gpu)
if EXEMPLES_PRIVES:
    # Nom hors du motif *_asr_*.json de rapport_asr.py, et ignoré par .gitignore (*_PRIVE*).
    chemin_prive = f"{SORTIE}/{horodatage[:10]}_asr-{FAMILLE}_PRIVE.json"
    json.dump(EXEMPLES_PRIVES, open(chemin_prive, "w", encoding="utf-8"), ensure_ascii=False, indent=1)
    print("exemples des corpus privés →", chemin_prive, "| à garder dans le dataset Kaggle privé, JAMAIS dans le dépôt")
print("À déposer dans bench/resultats/ du dépôt, puis : python bench/rapport_asr.py")